# Recommendations

Purchase history with ratings, from `data/sample_history.json`: made-up data shaped like the item service's `History` (with its `Item`) and `Review` models. Regenerate or resize it with `python3 data/make_sample_history.py [count]`.

In [1]:
import json
from pathlib import Path

import pandas as pd

raw = json.loads(Path("data/sample_history.json").read_text(encoding="utf-8"))
print(f"{len(raw)} purchases, {sum(h['review'] is not None for h in raw)} rated")

60 purchases, 48 rated


## One row per purchase

The item's and the review's fields flattened next to the history's; `score` and `content` are empty for purchases not rated yet.

In [2]:
history = pd.json_normalize(raw).drop(columns=["item.id", "review", "review.id", "review.history_id"], errors="ignore")
history.columns = [c.split(".", 1)[1] if c.startswith(("item.", "review.")) else c for c in history.columns]
history = history.rename(columns={"id": "history_id"})
for col in ["time", "created_at", "deadline_at", "buy_at", "schedule"]:
    history[col] = pd.to_datetime(history[col])
history["score"] = history["score"].astype("Int64")

history[["time", "owner", "item_type", "item_name", "shop_name", "score", "content"]].head(10)

,time,owner,item_type,item_name,shop_name,score,content
0,2026-11-24 22:17:00+07:00,lan,food_and_drink,Trà đào cam sả,Highlands Coffee,<NA>,NaN
1,2026-11-20 19:43:20+07:00,lan,food_and_drink,Trà sữa trân châu đường đen,Phúc Long,5,"Hoàn hảo, không chê được gì"
2,2026-11-10 07:09:07+07:00,lan,food_and_drink,Cà phê muối,Cà Phê Muối Chú Long,3,"Bình thường, không có gì đặc biệt"
3,2026-11-09 15:45:20+07:00,minh,restaurant,Lẩu Thái Tomyum,None,5,"Hoàn hảo, không chê được gì"
4,2026-11-07 16:26:20+07:00,lan,clothes,Túi tote canvas,Shopee,4,"Ngon, sẽ quay lại"
5,2026-10-30 07:21:44+07:00,lan,food_and_drink,Cà phê muối,Cà Phê Muối Chú Long,4,"Ngon, sẽ quay lại"
6,2026-10-26 21:07:22+07:00,lan,restaurant,Bún chả Hương Liên,None,4,"Ngon, sẽ quay lại"
7,2026-10-17 21:16:21+07:00,lan,food_and_drink,Trà sữa trân châu đường đen,Phúc Long,<NA>,NaN
8,2026-10-17 11:17:43+07:00,minh,clothes,Áo hoodie nỉ,Uniqlo Đồng Khởi,4,"Ngon, sẽ quay lại"
9,2026-10-15 11:14:18+07:00,lan,restaurant,Sushi Hokkaido Sachi,None,3,"Ổn, nhưng có chỗ khác ngon hơn"


## Ratings

The rated purchases only, and each person's average score per item: the user–item matrix a recommender starts from.

In [3]:
ratings = history.dropna(subset=["score"])[["owner", "item_type", "item_name", "score"]]
ratings.groupby(["owner", "item_type"])["score"].agg(["count", "mean"]).round(2)

count  mean
owner item_type                  
lan   clothes            11  3.73
      food_and_drink     12  4.08
      others              2   4.0
      restaurant          6   4.0
minh  clothes             8  3.75
      food_and_drink      5   4.0
      others              2   2.5
      restaurant          2   5.0

In [4]:
matrix = ratings.pivot_table(index="owner", columns="item_name", values="score", aggfunc="mean").round(1)
matrix

item_name,Bánh flan caramel,Bánh tráng trộn,Bún chả Hương Liên,Cà phê muối,Giày Air Force 1,Gogi House,Gấu bông Capybara,Lẩu Thái Tomyum,Matcha latte,Nến thơm vanilla,...,Son kem lì,Sushi Hokkaido Sachi,Trà sữa trân châu đường đen,Trà đào cam sả,Túi tote canvas,Váy hoa nhí,Áo hoodie nỉ,Áo sơ mi linen,Áo thun oversize,Ốp lưng điện thoại
owner,,,,,,,,,,,,,,,,,,,,,
lan,<NA>,3.5,4.0,3.5,3.0,5.0,<NA>,3.0,4.5,4.0,...,<NA>,3.0,4.8,<NA>,4.0,4.0,4.5,3.0,4.0,4.0
minh,3.0,<NA>,<NA>,<NA>,4.5,<NA>,3.0,5.0,4.0,<NA>,...,2.0,<NA>,4.0,4.0,2.0,<NA>,4.0,3.0,4.0,<NA>


## Language: Vietnamese or English

Vietnamese is told apart by its letters: â ă đ ê ô ơ ư and the vowels carrying a tone mark. A text with any of them is Vietnamese, anything else English. (An English word with é, like café, counts as Vietnamese.)

In [5]:
import re
import unicodedata

VIETNAMESE = re.compile(
    r"[àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]", re.IGNORECASE
)


def language(text: str) -> str:
    """'vi' when the text has a Vietnamese letter, else 'en'. NFC first, so a letter typed as
    base + combining mark (as some keyboards do) is one character again."""
    return "vi" if VIETNAMESE.search(unicodedata.normalize("NFC", text)) else "en"


assert language("Trà sữa trân châu") == "vi" and language("ĐÀ NẴNG") == "vi"
assert language("Pizza 4P's") == "en" and language("So good, thank you!") == "en"

history["name_lang"] = history["item_name"].map(language)
history["content_lang"] = history["content"].map(lambda t: language(t) if isinstance(t, str) and t else None)
pd.concat({"item_name": history["name_lang"].value_counts(), "content": history["content_lang"].value_counts()}, axis=1)

,item_name,content
vi,51,35.0
en,9,NaN


## Embeddings

Each text goes to its language's model: [PhoBERT](https://huggingface.co/vinai/phobert-base-v2) for Vietnamese, [ModernBERT](https://huggingface.co/answerdotai/ModernBERT-base) for English. Its vector is the mean of the model's last hidden states over the text's tokens (768 numbers for both).

PhoBERT was trained on word-segmented text (`trà_sữa trân_châu`), so Vietnamese is segmented first, with underthesea. The two models' vectors are in different spaces: compare a Vietnamese text only with other Vietnamese ones, and an English one with English ones.

In [6]:
import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer
from underthesea import word_tokenize

MODELS = {"vi": "vinai/phobert-base-v2", "en": "answerdotai/ModernBERT-base"}
loaded = {}  # language -> (tokenizer, model), downloaded the first time


def model_for(lang):
    if lang not in loaded:
        loaded[lang] = AutoTokenizer.from_pretrained(MODELS[lang]), AutoModel.from_pretrained(MODELS[lang]).eval()
    return loaded[lang]


@torch.no_grad()
def embed(texts, lang, batch_size=16):
    """One vector per text, all in the one language"""
    if lang == "vi":
        texts = [word_tokenize(t, format="text") for t in texts]
    tokenizer, model = model_for(lang)
    vectors = []
    for start in range(0, len(texts), batch_size):
        # PhoBERT takes at most 256 tokens
        batch = tokenizer(texts[start : start + batch_size], padding=True, truncation=True, max_length=256, return_tensors="pt")
        hidden = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).last_hidden_state
        mask = batch["attention_mask"].unsqueeze(-1).to(hidden.dtype)
        vectors.append((hidden * mask).sum(1) / mask.sum(1))
    return torch.cat(vectors).numpy()


def embed_by_language(texts):
    """Each text embedded by its own language's model: (vectors in the texts' order, languages)"""
    texts = list(texts)
    langs = [language(t) for t in texts]
    vectors = [None] * len(texts)
    for lang in sorted(set(langs)):
        at = [i for i, l in enumerate(langs) if l == lang]
        for i, vector in zip(at, embed([texts[i] for i in at], lang)):
            vectors[i] = vector
    return np.stack(vectors), langs

Every distinct item name, and every review that has words in it:

In [7]:
items = history.drop_duplicates("item_name")[["item_name", "item_type"]].reset_index(drop=True)
item_vectors, items["lang"] = embed_by_language(items["item_name"])

reviews = history[history["content"].fillna("") != ""][["history_id", "owner", "item_name", "score", "content"]]
reviews = reviews.reset_index(drop=True)
review_vectors, reviews["lang"] = embed_by_language(reviews["content"])

print("items:", item_vectors.shape, items["lang"].value_counts().to_dict())
print("reviews:", review_vectors.shape, reviews["lang"].value_counts().to_dict())

items: (26, 768) {'vi': 22, 'en': 4}
reviews: (35, 768) {'vi': 35}


## Item type similarity

How close two categories are, from a description of each (ModernBERT, English). The descriptions share one pattern ("Something to … : … such as …"), so they differ only in what they're about. BERT puts all sentences fairly close together, so the similarities are rescaled: food_and_drink against clothes, which have nothing to do with each other, becomes 0 (anything less, too), and a category against itself is 1.

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

CATEGORY_DESCRIPTIONS = {
    "food_and_drink": "Something to eat and drink: food and drinks such as milk tea, coffee, snacks, cakes and desserts.",
    "restaurant": "Somewhere to eat and drink: a restaurant serving food and drinks such as noodles, hotpot, sushi and barbecue.",
    "clothes": "Something to wear: clothes and fashion such as shirts, dresses, jeans, jackets, shoes and bags.",
    "others": "Something else to buy: gifts and things such as cosmetics, electronics, books, decorations and accessories.",
}
UNRELATED = ("food_and_drink", "clothes")  # the pair that counts as 0

categories = list(CATEGORY_DESCRIPTIONS)
raw = cosine_similarity(embed(list(CATEGORY_DESCRIPTIONS.values()), "en"))
floor = raw[categories.index(UNRELATED[0]), categories.index(UNRELATED[1])]
type_similarity = pd.DataFrame(np.clip((raw - floor) / (1 - floor), 0, 1), index=categories, columns=categories)
type_similarity.round(2)

,food_and_drink,restaurant,clothes,others
food_and_drink,1.00,0.49,0.0,0.16
restaurant,0.49,1.00,0.0,0.00
clothes,0.00,0.00,1.0,0.40
others,0.16,0.00,0.4,1.00


## Item similarity: name and type

Two items' similarity is their names' similarity and their types' similarity, weighted by `TYPE_WEIGHT`. At 0.5 that's what the cosine of each item's name vector and type vector, both normalized and concatenated, would give. Names are still compared only within one language (the two models' vectors don't mix), so items named in different languages aren't scored against each other.

In [9]:
TYPE_WEIGHT = 0.5  # how much the item type counts, against the name

name_similarity = cosine_similarity(item_vectors)
type_of = items["item_type"].to_numpy()
type_pairs = type_similarity.loc[type_of, type_of].to_numpy()
same_language = items["lang"].to_numpy()[:, None] == items["lang"].to_numpy()[None, :]
item_similarity = np.where(same_language, (1 - TYPE_WEIGHT) * name_similarity + TYPE_WEIGHT * type_pairs, np.nan)


def similar_items(item, k=5):
    """The k items most like `item`, with what their score is made of. `item` is a row with an
    item_name and an item_type: from items or history, or a new one that isn't in either."""
    vector, (lang,) = embed_by_language([item["item_name"]])
    name = cosine_similarity(vector, item_vectors)[0]
    type_ = type_similarity.loc[item["item_type"], type_of].to_numpy()
    score = np.where(items["lang"] == lang, (1 - TYPE_WEIGHT) * name + TYPE_WEIGHT * type_, np.nan)
    return (
        items[["item_name", "item_type"]]
        .assign(name=name.round(3), type=type_.round(2), similarity=score.round(3))
        [items["item_name"] != item["item_name"]]
        .dropna(subset=["similarity"])
        .nlargest(k, "similarity")
    )


similar_items(history.iloc[1])  # a purchase: Trà sữa trân châu đường đen

,item_name,item_type,name,type,similarity
0,Trà đào cam sả,food_and_drink,0.668,1.0,0.834
2,Cà phê muối,food_and_drink,0.584,1.0,0.792
17,Bánh flan caramel,food_and_drink,0.559,1.0,0.779
18,Sinh tố bơ,food_and_drink,0.490,1.0,0.745
14,Bánh tráng trộn,food_and_drink,0.456,1.0,0.728


In [10]:
# Something not bought yet works too
similar_items(pd.Series({"item_name": "Bún bò Huế", "item_type": "restaurant"}))

,item_name,item_type,name,type,similarity
5,Bún chả Hương Liên,restaurant,0.666,1.00,0.833
3,Lẩu Thái Tomyum,restaurant,0.622,1.00,0.811
20,Phở Thìn,restaurant,0.544,1.00,0.772
14,Bánh tráng trộn,food_and_drink,0.673,0.49,0.580
17,Bánh flan caramel,food_and_drink,0.599,0.49,0.542
